In [1]:
# Import sys so we can tell Python where to look for files.
import sys

# Import Path to work with folder locations.
from pathlib import Path

# Get the notebook's current working directory.
current_dir = Path.cwd()

# Check whether Python is currently inside the notebooks folder.
# If so, go up one level to reach the main project folder.
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

# Add the main project folder to Python's import search paths.
# This allows Python to find the scripts folder.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import the Week 3 and Week 6 extractors
# from the signal_extractor.py file inside scripts.
from scripts.signal_extractor import (
    EntityExtractor,
    SignalExtractor,
    SIGNAL_TAXONOMY
)

# Initialize the Week 3 extractor.
entity_extractor = EntityExtractor()

# Initialize the Week 6 extractor using our taxonomy
# and the existing Week 3 extraction methods.
signal_extractor = SignalExtractor(
    taxonomy=SIGNAL_TAXONOMY,
    entity_extractor=entity_extractor
)

# Confirm that everything loaded successfully.
print("Week 3 EntityExtractor initialized successfully!")
print("Week 6 SignalExtractor initialized successfully!")

Week 3 EntityExtractor initialized successfully!
Week 6 SignalExtractor initialized successfully!


In [2]:
# Create a sample property listing.
# This is a fictional listing used to check whether
# our extractor correctly identifies different features.

sample_listing = {
    "L_ListingID": "TEST001",

    "L_Remarks": (
        "Beautiful renovated 3 bedroom, 2 bathroom home "
        "with 2,100 square feet of living space. "
        "Priced at $450,000. "
        "Features a swimming pool, fireplace, and attached garage. "
        "Located on a quiet cul-de-sac near schools. "
        "Seller financing available."
    )
}

# Extract all signals from the sample listing.
sample_result = signal_extractor.extract_signals(sample_listing)

# Display the result in an easy-to-read JSON format.
# indent=4 adds spaces so the output is easier to read.
import json

print(json.dumps(sample_result, indent=4))

{
    "listing_id": "TEST001",
    "entities": {
        "bedrooms": 3,
        "bathrooms": 2,
        "price": 450000,
        "sqft": 2100,
        "amenities": [
            "pool",
            "fireplace",
            "garage"
        ]
    },
    "amenities": [
        "pool",
        "fireplace",
        "garage"
    ],
    "condition_keywords": [
        "renovated"
    ],
    "financing_terms": [
        "seller_financing"
    ],
    "location_features": [
        "cul_de_sac",
        "near_schools"
    ]
}


In [3]:
# ============================================================
# STEP 3: LOAD REAL ESTATE LISTINGS FROM MYSQL
# ============================================================

# Import libraries for database connections and data handling.
import os
import mysql.connector
import pandas as pd

# Import getpass to securely enter the database password.
# The password will not be displayed when typed.
from getpass import getpass

# Connect to the MySQL database.
# The password is entered privately when the cell runs,
# rather than being saved directly in the notebook.
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("Enter MySQL password: "),
    database="real_estate"
)

# Retrieve all real estate listings.
# Include structured fields for evaluating extraction accuracy.
# No LIMIT is used because Week 6 requires the entire table.
query = """
SELECT
    L_ListingID,
    L_Address,
    L_City,
    L_Keyword2 AS beds,
    LM_Dec_3 AS baths,
    L_SystemPrice AS price,
    L_Remarks
FROM rets_property
"""

# Load the SQL results into a pandas DataFrame.
try:
    df = pd.read_sql(query, conn)

finally:
    # Close the connection even if loading fails.
    conn.close()

# Display the number of listings loaded.
print(f"Total listings loaded: {len(df):,}")

# Display the first five records.
display(df.head())

# Display the available column names.
print("\nAvailable columns:")
print(df.columns.tolist())

C:\Users\luoxu\AppData\Local\Temp\ipykernel_30556\84944378.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Total listings loaded: 55,212


,L_ListingID,L_Address,L_City,beds,baths,price,L_Remarks
0,1118422731,1461 Laurel Way,Beverly Hills,4.0,5.0,3950000,Opportunity to reimagine a Classic 70's Archit...
1,1118398412,582 Pasture Ave,Lathrop,4.0,3.0,775000,Step into comfort and functionality with this ...
2,1118383115,14947 Dickens Street 3,Sherman Oaks,2.0,2.0,750000,WOW! NOW REDUCED AND THE SELLER MAY FINANCE! C...
3,1118381441,349 E Gobind Blvd,Mountain House,5.0,4.0,1122426,Plan 3 features an open floor plan with covere...
4,1118372790,515 E Fiorino Street,Mountain House,5.0,3.0,1128292,Plan 2 features an open floor plan with covere...



Available columns:
['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price', 'L_Remarks']


In [ ]:

# ============================================================
# STEP 4: EXTRACT SIGNALS FROM ALL LISTINGS
# ============================================================

# Create an empty list to store the extracted signals.
processed_listings = []

# Convert the DataFrame into a list of dictionaries.
# Each dictionary represents one property listing.
listing_records = df.to_dict(orient="records")

# Loop through every listing.
for listing in listing_records:

    # Run our Week 6 SignalExtractor on the listing.
    # This extracts bedrooms, bathrooms, price, square footage,
    # amenities, condition, financing, and location features.
    signals = signal_extractor.extract_signals(listing)

    # Add the extracted information to our results list.
    processed_listings.append(signals)

# Convert the extracted results into a pandas DataFrame.
# This makes it easier to inspect and analyze the information.
signals_df = pd.DataFrame(processed_listings)

# Display the number of listings processed.
# Display the first five results.
display(signals_df.head())


Successfully processed 55,212 listings.


,listing_id,entities,amenities,condition_keywords,financing_terms,location_features
0,1118422731,"{'bedrooms': None, 'bathrooms': None, 'price':...",[],[],[],[cul_de_sac]
1,1118398412,"{'bedrooms': None, 'bathrooms': 3, 'price': No...","[patio, parking, garage]",[],[],[]
2,1118383115,"{'bedrooms': None, 'bathrooms': None, 'price':...","[laundry, fireplace, patio, updated_kitchen]",[updated],[],[]
3,1118381441,"{'bedrooms': None, 'bathrooms': None, 'price':...",[patio],[],[],[]
4,1118372790,"{'bedrooms': None, 'bathrooms': None, 'price':...",[patio],[],[],[]


In [5]:

# ============================================================
# STEP 5: EXPORT EXTRACTED LISTING SIGNALS
# ============================================================

# Import the libraries needed for JSON export and file paths.
import json
from pathlib import Path

# Use the project_root variable from our first notebook cell.
#
# This saves the file in the project's processed folder,
# regardless of whether the notebook is inside notebooks/.
output_dir = project_root / "processed"

# Create the processed folder if it does not exist.
output_dir.mkdir(parents=True, exist_ok=True)

# Set the final output filename.
output_file = output_dir / "week6_listing_signals.json"

# Save every extracted listing as JSON.
#
# indent=2 makes the output readable.
# ensure_ascii=False preserves special characters.
# default=str handles database values that JSON may not recognize.
with open(output_file, "w", encoding="utf-8") as file:
    json.dump(
        processed_listings,
        file,
        indent=2,
        ensure_ascii=False,
        default=str
    )

# Display confirmation that the file was saved.
print(f"Saved {len(processed_listings):,} listings.")
print(f"File location: {output_file.resolve()}")


Saved 55,212 listings.
File location: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week6_listing_signals.json


In [ ]:

# ============================================================
# STEP 7: PREPARE MANUAL EXTRACTION EVALUATION
# ============================================================

# Randomly select up to 100 listings for manual evaluation.
# random_state=42 ensures we get the same sample each time.
evaluation_sample = df.sample(
    n=min(100, len(df)),
    random_state=42
)

# Create an empty list to store evaluation records.
evaluation_records = []

# Process each listing in our evaluation sample.
for listing in evaluation_sample.to_dict(orient="records"):

    # Generate predicted signals using our extractor.
    predictions = signal_extractor.extract_signals(listing)

    # Store the original description, reference database values,
    # and signals predicted by our extractor.
    evaluation_records.append({
        "listing_id": listing["L_ListingID"],
        "remarks": listing["L_Remarks"],

        "database_beds": listing["beds"],
        "database_baths": listing["baths"],
        "database_price": listing["price"],

        "predicted_entities": json.dumps(predictions["entities"]),
        "predicted_amenities": json.dumps(predictions["amenities"]),
        "predicted_condition": json.dumps(predictions["condition_keywords"]),
        "predicted_financing": json.dumps(predictions["financing_terms"]),
        "predicted_location": json.dumps(predictions["location_features"]),

        # Fill in these columns manually after reviewing the remarks.
        # Use JSON lists such as ["pool", "garage"].
        "ground_truth_amenities": "",
        "ground_truth_condition": "",
        "ground_truth_financing": "",
        "ground_truth_location": ""
    })

# Convert the evaluation records into a DataFrame.
evaluation_df = pd.DataFrame(evaluation_records)

# Define the evaluation file location.
evaluation_file = output_dir / "week6_signal_evaluation.csv"

# Save the sample to a CSV file for manual review.
evaluation_df.to_csv(evaluation_file, index=False)

# Display the first five records.
display(evaluation_df.head())

# Confirm the file was saved.
print(f"Evaluation file saved to: {evaluation_file.resolve()}")


,listing_id,remarks,database_beds,database_baths,database_price,predicted_entities,predicted_amenities,predicted_condition,predicted_financing,predicted_location,ground_truth_amenities,ground_truth_condition,ground_truth_financing,ground_truth_location
0,1183036503,Welcome to this beautifully appointed Spanish-...,3.0,3.0,749900,"{""bedrooms"": 3, ""bathrooms"": 2.5, ""price"": nul...","[""patio"", ""garage"", ""laundry"", ""balcony""]",[],[],[],,,,
1,1190152982,Back on the market with New Pergo Flooring and...,3.0,2.0,574900,"{""bedrooms"": 2, ""bathrooms"": 1, ""price"": null,...","[""updated_kitchen"", ""patio"", ""laundry"", ""parki...","[""updated"", ""move_in_ready""]",[],[],,,,
2,1177518498,**TWO FLAT PARCELS ARE INCLUDED IN THE SALES P...,1.0,2.0,234900,"{""bedrooms"": 1, ""bathrooms"": 1, ""price"": null,...","[""parking""]",[],[],[],,,,
3,1184980333,Priced to sell !! Bright and Spacious 3BR + 2...,3.0,3.0,895000,"{""bedrooms"": 3, ""bathrooms"": 2.5, ""price"": nul...","[""balcony"", ""laundry"", ""parking""]",[],[],[],,,,
4,1177243676,"Located in San Martin, this gorgeous and peace...",4.0,3.0,1688000,"{""bedrooms"": 4, ""bathrooms"": 2, ""price"": null,...","[""patio"", ""pool"", ""fireplace"", ""laundry"", ""gar...",[],[],[],,,,


Evaluation file saved to: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week6_signal_evaluation.csv


In [29]:

# ============================================================
# RESTORE MANUAL REVIEW CATEGORY CONFIGURATION
# ============================================================

# The evaluation code reused the variable "categories".
# Restore its original format for the manual review widget.
#
# Each tuple contains:
# 1. The displayed category name
# 2. The original predicted-signals column
# 3. The manually corrected ground-truth column

categories = [
    ("Amenities", "predicted_amenities", "ground_truth_amenities"),
    ("Condition", "predicted_condition", "ground_truth_condition"),
    ("Financing", "predicted_financing", "ground_truth_financing"),
    ("Location", "predicted_location", "ground_truth_location")
]

print("Manual review category configuration restored!")

Manual review category configuration restored!


In [30]:

# ============================================================
# STEP 8: INTERACTIVE MANUAL SIGNAL EVALUATION
# ============================================================

# Import libraries for interactive notebook controls.
import ipywidgets as widgets
import pandas as pd
import json
import html

from IPython.display import display


# ------------------------------------------------------------
# 1. LOAD EXISTING EVALUATION DATA
# ------------------------------------------------------------

# If a saved evaluation file already exists, load it.
# This preserves any manual corrections from earlier sessions.
#
# Otherwise, start with the original evaluation DataFrame.
if evaluation_file.exists():
    manual_df = pd.read_csv(
        evaluation_file,
        dtype={"listing_id": str}
    ).fillna("")
else:
    manual_df = evaluation_df.copy().fillna("")


# Create a column to track which listings have been reviewed.
# This allows us to distinguish reviewed empty answers
# from listings that have not been checked yet.
if "reviewed" not in manual_df.columns:
    manual_df["reviewed"] = False

# Convert saved review status into Boolean values.
manual_df["reviewed"] = (
    manual_df["reviewed"].astype(str).str.lower() == "true"
)


# ------------------------------------------------------------
# 2. DEFINE SIGNAL CATEGORIES
# ------------------------------------------------------------

# These are the four categories we want to review.
#
# Each category includes:
# - The display name
# - The model prediction column
# - The manual ground-truth column

categories = [
    ("Amenities", "predicted_amenities", "ground_truth_amenities"),
    ("Condition", "predicted_condition", "ground_truth_condition"),
    ("Financing", "predicted_financing", "ground_truth_financing"),
    ("Location", "predicted_location", "ground_truth_location")
]


# ------------------------------------------------------------
# 3. CREATE INTERFACE COMPONENTS
# ------------------------------------------------------------

# Store the position of the listing currently being reviewed.
current_index = 0

# Create a title showing the current listing number.
title_widget = widgets.HTML()

# Create a box for displaying the property description.
#
# HTML automatically wraps long text and preserves line breaks.
description_widget = widgets.HTML()

# Create a dictionary to store the editable fields.
input_widgets = {}

# Create the interface elements for each category.
category_widgets = []

for label, predicted_col, truth_col in categories:

    # Show the original model prediction.
    prediction_label = widgets.HTML()

    # Create a text box where you can enter corrected labels.
    #
    # Textarea allows longer lists and wraps text automatically.
    input_box = widgets.Textarea(
        placeholder="Enter correct signals separated by commas",
        layout=widgets.Layout(
            width="100%",
            height="55px"
        )
    )

    # Store the widgets so we can update them later.
    input_widgets[truth_col] = {
        "prediction": prediction_label,
        "input": input_box,
        "predicted_col": predicted_col
    }

    # Group the prediction and input box together.
    category_box = widgets.VBox([
        widgets.HTML(f"<b>{label}</b>"),
        prediction_label,
        input_box
    ])

    category_widgets.append(category_box)


# Create navigation buttons.
previous_button = widgets.Button(
    description="Previous",
    button_style="",
    icon="arrow-left"
)

save_button = widgets.Button(
    description="Save Review",
    button_style="success",
    icon="check"
)

next_button = widgets.Button(
    description="Next",
    button_style="info",
    icon="arrow-right"
)

# Create a label for showing save confirmation.
status_widget = widgets.HTML()


# ------------------------------------------------------------
# 4. DISPLAY A LISTING
# ------------------------------------------------------------

def show_listing(index):
    """
    Update the interface to show the selected listing.

    This function changes the displayed text and input values
    without recreating the widgets.
    """

    global current_index

    # Update the current listing position.
    current_index = index

    # Retrieve the selected listing.
    row = manual_df.iloc[index]

    # Display listing information.
    title_widget.value = (
        f"<h3>Listing {index + 1} of {len(manual_df)}</h3>"
        f"<b>Listing ID:</b> {html.escape(str(row['listing_id']))}"
    )

    # Convert the listing description into safe HTML text.
    #
    # html.escape() prevents special characters in remarks
    # from being interpreted as HTML.
    remarks = html.escape(str(row["remarks"]))

    # Display the complete description in a scrollable box.
    description_widget.value = f"""
    <div style="
        white-space: pre-wrap;
        overflow-wrap: anywhere;
        max-height: 300px;
        overflow-y: auto;
        padding: 12px;
        border: 1px solid #cccccc;
        border-radius: 6px;
        line-height: 1.6;
    ">
        {remarks}
    </div>
    """

    # Update every signal category.
    for label, predicted_col, truth_col in categories:

        # Retrieve the predicted signal list.
        predicted = json.loads(row[predicted_col])

        # Display the original model prediction.
        input_widgets[truth_col]["prediction"].value = (
            f"<span style='color:gray;'>"
            f"Predicted: {html.escape(', '.join(predicted) or 'None')}"
            f"</span>"
        )

        # Retrieve existing manual corrections.
        saved_value = row[truth_col]

        # If previously reviewed, display the saved correction.
        if row["reviewed"] and saved_value:
            labels = json.loads(saved_value)

        # If previously reviewed with no signals, display blank.
        elif row["reviewed"]:
            labels = []

        # Otherwise, start with the model's prediction.
        else:
            labels = predicted

        # Populate the editable input box.
        input_widgets[truth_col]["input"].value = ", ".join(labels)

    # Disable buttons when reaching the first or last listing.
    previous_button.disabled = index == 0
    next_button.disabled = index == len(manual_df) - 1

    # Display whether this listing has already been reviewed.
    if row["reviewed"]:
        status_widget.value = "<b style='color:green;'>Previously reviewed</b>"
    else:
        status_widget.value = "Not yet reviewed"


# ------------------------------------------------------------
# 5. SAVE MANUAL CORRECTIONS
# ------------------------------------------------------------

def save_review(button=None):
    """
    Save corrected labels into the evaluation DataFrame
    and export them to CSV.
    """

    # Retrieve the row corresponding to the current listing.
    row_index = manual_df.index[current_index]

    # Save corrections for every signal category.
    for label, predicted_col, truth_col in categories:

        # Get the text entered in the corresponding box.
        entered_text = input_widgets[truth_col]["input"].value

        # Split comma-separated text into individual labels.
        #
        # Example:
        # "pool, garage" becomes ["pool", "garage"].
        labels = [
            item.strip()
            for item in entered_text.split(",")
            if item.strip()
        ]

        # Save labels as JSON text for later evaluation.
        manual_df.at[row_index, truth_col] = json.dumps(labels)

    # Mark this listing as reviewed.
    manual_df.at[row_index, "reviewed"] = True

    # Save progress to the evaluation CSV file.
    manual_df.to_csv(evaluation_file, index=False)

    # Show confirmation.
    status_widget.value = "<b style='color:green;'>Review saved!</b>"


# ------------------------------------------------------------
# 6. NAVIGATION BUTTONS
# ------------------------------------------------------------

def go_previous(button):
    # Save the current corrections before moving backward.
    save_review()

    # Display the previous listing.
    if current_index > 0:
        show_listing(current_index - 1)


def go_next(button):
    # Save the current corrections before moving forward.
    save_review()

    # Display the next listing.
    if current_index < len(manual_df) - 1:
        show_listing(current_index + 1)


# Connect each button to its corresponding function.
previous_button.on_click(go_previous)
save_button.on_click(save_review)
next_button.on_click(go_next)


# ------------------------------------------------------------
# 7. ASSEMBLE AND DISPLAY THE INTERFACE
# ------------------------------------------------------------

# Arrange the navigation buttons horizontally.
navigation = widgets.HBox([
    previous_button,
    save_button,
    next_button
])

# Combine all widgets into one interface.
review_interface = widgets.VBox(
    [
        title_widget,
        description_widget,
        *category_widgets,
        navigation,
        status_widget
    ],
    layout=widgets.Layout(
        width="100%"
    )
)

# Display the interface only once.
display(review_interface)

# Resume at the first listing that has not yet been reviewed.
unreviewed = manual_df.index[~manual_df["reviewed"]]

if len(unreviewed) > 0:
    starting_index = manual_df.index.get_loc(unreviewed[0])
else:
    starting_index = 0

# Show the selected listing.
show_listing(starting_index)



### Week 6: Signal Taxonomy — Quick Reference

**Amenities (12)**
- `pool`, `garage`, `parking`, `fireplace`
- `patio`, `deck`, `balcony`, `basement`
- `solar`, `air_conditioning`, `laundry`, `updated_kitchen`

**Condition (5)**
- `renovated` — Remodeled or renovated
- `updated` — Updated or upgraded
- `new_construction` — Newly built
- `move_in_ready` — Turnkey or ready to move in
- `fixer_upper` — Needs repairs or renovation

**Financing (4)**
- `seller_financing` — Owner/seller financing
- `assumable_mortgage` — Assumable loan
- `cash_only` — Cash buyers only
- `lease_to_own` — Rent-to-own option

**Location (7)**
- `waterfront` — Waterfront, lakefront, riverfront
- `cul_de_sac` — Cul-de-sac
- `near_schools` — Close to schools
- `near_parks` — Close to parks
- `corner_lot` — Corner lot
- `gated_community` — Gated neighborhood
- `near_shopping` — Close to shopping

**Manual Evaluation Rules**
- Use only labels listed above.
- Only label features supported by the description.
- Do not count negated features (e.g., "no pool").
- Leave the input blank if no signals apply.
- Separate multiple labels with commas.

**Total: 28 signals**


In [12]:
   # ============================================================
# STEP 9: KEEP ONLY MANUALLY REVIEWED LISTINGS
# ============================================================

# Load the evaluation file containing your manual corrections.
# This preserves the work you've already completed.
manual_df = pd.read_csv(evaluation_file).fillna("")

# Convert the reviewed column to Boolean values.
# This ensures that "True" and "False" are interpreted correctly.
manual_df["reviewed"] = (
    manual_df["reviewed"].astype(str).str.lower() == "true"
)

# Select only listings that have been manually reviewed.
# The remaining unreviewed listings are excluded.
reviewed_df = manual_df[manual_df["reviewed"]].copy()

# Save your reviewed listings to a separate file.
# This preserves your original 100-listing evaluation file.
reviewed_file = output_dir / "week6_reviewed_evaluation.csv"

reviewed_df.to_csv(reviewed_file, index=False)

# Display how many listings were reviewed.
print(f"Original evaluation sample: {len(manual_df)} listings")
print(f"Manually reviewed: {len(reviewed_df)} listings")
print(f"Saved to: {reviewed_file.resolve()}")

# Preview your reviewed records.
display(reviewed_df.head())

Original evaluation sample: 100 listings
Manually reviewed: 25 listings
Saved to: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week6_reviewed_evaluation.csv


,listing_id,remarks,database_beds,database_baths,database_price,predicted_entities,predicted_amenities,predicted_condition,predicted_financing,predicted_location,ground_truth_amenities,ground_truth_condition,ground_truth_financing,ground_truth_location,reviewed
0,1183036503,Welcome to this beautifully appointed Spanish-...,3.0,3.0,749900,"{""bedrooms"": 3, ""bathrooms"": 2.5, ""price"": nul...","[""patio"", ""garage"", ""laundry"", ""balcony""]",[],[],[],"[""patio"", ""garage"", ""laundry"", ""balcony""]",[],[],[],True
1,1190152982,Back on the market with New Pergo Flooring and...,3.0,2.0,574900,"{""bedrooms"": 2, ""bathrooms"": 1, ""price"": null,...","[""updated_kitchen"", ""patio"", ""laundry"", ""parki...","[""updated"", ""move_in_ready""]",[],[],"[""updated_kitchen"", ""patio"", ""laundry"", ""parki...","[""updated"", ""move_in_ready""]",[],[],True
2,1177518498,**TWO FLAT PARCELS ARE INCLUDED IN THE SALES P...,1.0,2.0,234900,"{""bedrooms"": 1, ""bathrooms"": 1, ""price"": null,...","[""parking""]",[],[],[],"[""parking""]",[],[],[],True
3,1184980333,Priced to sell !! Bright and Spacious 3BR + 2...,3.0,3.0,895000,"{""bedrooms"": 3, ""bathrooms"": 2.5, ""price"": nul...","[""balcony"", ""laundry"", ""parking""]",[],[],[],"[""balcony"", ""laundry"", ""parking""]",[],[],[],True
4,1177243676,"Located in San Martin, this gorgeous and peace...",4.0,3.0,1688000,"{""bedrooms"": 4, ""bathrooms"": 2, ""price"": null,...","[""patio"", ""pool"", ""fireplace"", ""laundry"", ""gar...",[],[],[],"[""patio"", ""pool"", ""fireplace"", ""laundry"", ""gar...",[],[],[],True


In [13]:

# ============================================================
# STEP 10: EVALUATE FREE-TEXT SIGNAL EXTRACTION
# ============================================================

# Import libraries for data processing and evaluation.
import pandas as pd
import json

# Import classification metrics from scikit-learn.
from sklearn.metrics import precision_score, recall_score, f1_score

# MultiLabelBinarizer converts lists of signals into binary values.
# This allows us to evaluate multiple signals for each listing.
from sklearn.preprocessing import MultiLabelBinarizer


# ------------------------------------------------------------
# 1. DEFINE THE SIGNAL CATEGORIES
# ------------------------------------------------------------

# Each category contains two columns:
# - Predicted signals generated by our SignalExtractor.
# - Ground-truth signals manually verified by us.
categories = {
    "Amenities": ("predicted_amenities", "ground_truth_amenities"),
    "Condition": ("predicted_condition", "ground_truth_condition"),
    "Financing": ("predicted_financing", "ground_truth_financing"),
    "Location": ("predicted_location", "ground_truth_location")
}


# ------------------------------------------------------------
# 2. CREATE AN EMPTY LIST FOR EVALUATION RESULTS
# ------------------------------------------------------------

# We will store precision, recall, and F1-score
# for each signal category.
evaluation_results = []


# ------------------------------------------------------------
# 3. EVALUATE EACH SIGNAL CATEGORY
# ------------------------------------------------------------

for category, (pred_col, truth_col) in categories.items():

    # Convert JSON strings into Python lists.
    #
    # Example:
    # '["pool", "garage"]' becomes ["pool", "garage"].
    y_true = reviewed_df[truth_col].apply(json.loads).tolist()
    y_pred = reviewed_df[pred_col].apply(json.loads).tolist()

    # Create a MultiLabelBinarizer.
    #
    # This assigns one binary column to each possible signal.
    # A value of 1 means the signal is present.
    # A value of 0 means the signal is absent.
    mlb = MultiLabelBinarizer()

    # Learn which signals exist in the true and predicted data.
    mlb.fit(y_true + y_pred)

    # Convert our lists into binary matrices.
    true_binary = mlb.transform(y_true)
    pred_binary = mlb.transform(y_pred)

    # Check whether any signals exist in this category.
    #
    # If no signals appear in any reviewed listing,
    # precision, recall, and F1 cannot meaningfully
    # measure the extractor's ability to identify them.
    if len(mlb.classes_) == 0:

        precision = float("nan")
        recall = float("nan")
        f1 = float("nan")

    else:

        # Precision measures how many predicted signals were correct.
        #
        # Precision = True Positives / (True Positives + False Positives)
        precision = precision_score(
            true_binary,
            pred_binary,
            average="micro",
            zero_division=0
        )

        # Recall measures how many actual signals were detected.
        #
        # Recall = True Positives / (True Positives + False Negatives)
        recall = recall_score(
            true_binary,
            pred_binary,
            average="micro",
            zero_division=0
        )

        # F1-score balances precision and recall.
        #
        # A higher F1-score indicates better overall extraction.
        f1 = f1_score(
            true_binary,
            pred_binary,
            average="micro",
            zero_division=0
        )

    # Store the results for this category.
    evaluation_results.append({
        "Category": category,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Signals Evaluated": len(mlb.classes_)
    })


# ------------------------------------------------------------
# 4. DISPLAY EVALUATION RESULTS
# ------------------------------------------------------------

# Convert results into a DataFrame.
metrics_df = pd.DataFrame(evaluation_results)

# Display the number of listings evaluated.
print(f"Total manually reviewed listings: {len(reviewed_df)}")

# Display the results as percentages.
display(
    metrics_df.style.format({
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1 Score": "{:.2%}"
    })
)


Total manually reviewed listings: 25


,Category,Precision,Recall,F1 Score,Signals Evaluated
0,Amenities,100.00%,90.79%,95.17%,12
1,Condition,100.00%,78.57%,88.00%,5
2,Financing,0.00%,0.00%,0.00%,2
3,Location,100.00%,20.00%,33.33%,5


In [14]:

# ============================================================
# STEP 11: CALCULATE OVERALL EXTRACTION PERFORMANCE
# ============================================================

# Create counters for correct predictions and extraction errors.
total_tp = 0  # True positives: correctly identified signals.
total_fp = 0  # False positives: incorrectly predicted signals.
total_fn = 0  # False negatives: signals the extractor missed.

# Go through every signal category.
for category, (pred_col, truth_col) in categories.items():

    # Go through all 25 manually reviewed listings.
    for _, row in reviewed_df.iterrows():

        # Convert the stored JSON lists into Python sets.
        # Sets make it easy to compare predicted and actual signals.
        predicted = set(json.loads(row[pred_col]))
        actual = set(json.loads(row[truth_col]))

        # Count signals that appear in both sets.
        # These are correct predictions.
        total_tp += len(predicted & actual)

        # Count signals predicted but not actually present.
        total_fp += len(predicted - actual)

        # Count actual signals that the extractor missed.
        total_fn += len(actual - predicted)


# Calculate overall micro-averaged precision.
overall_precision = (
    total_tp / (total_tp + total_fp)
    if (total_tp + total_fp) > 0 else 0
)

# Calculate overall micro-averaged recall.
overall_recall = (
    total_tp / (total_tp + total_fn)
    if (total_tp + total_fn) > 0 else 0
)

# Calculate overall micro-averaged F1-score.
overall_f1 = (
    2 * overall_precision * overall_recall
    / (overall_precision + overall_recall)
    if (overall_precision + overall_recall) > 0 else 0
)


# Create a DataFrame summarizing the overall results.
overall_metrics_df = pd.DataFrame([{
    "Category": "Overall (Micro)",
    "Precision": overall_precision,
    "Recall": overall_recall,
    "F1 Score": overall_f1,
    "True Positives": total_tp,
    "False Positives": total_fp,
    "False Negatives": total_fn
}])


# Display the results as percentages.
display(
    overall_metrics_df.style.format({
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1 Score": "{:.2%}"
    })
)


,Category,Precision,Recall,F1 Score,True Positives,False Positives,False Negatives
0,Overall (Micro),100.00%,77.57%,87.37%,83,0,24


In [15]:

# ============================================================
# STEP 12: EVALUATE STRUCTURED FIELD EXTRACTION
# ============================================================

# Import libraries for reading and analyzing the data.
import pandas as pd
import json

# Create a copy of our 25 manually reviewed listings.
structured_df = reviewed_df.copy()

# Convert each predicted entity from JSON text to a dictionary.
# Example: '{"bedrooms": 3}' becomes {"bedrooms": 3}.
entities = structured_df["predicted_entities"].apply(json.loads)

# Extract the model's numeric predictions into separate columns.
structured_df["predicted_beds"] = entities.apply(
    lambda x: x.get("bedrooms")
)

structured_df["predicted_baths"] = entities.apply(
    lambda x: x.get("bathrooms")
)

structured_df["predicted_price"] = entities.apply(
    lambda x: x.get("price")
)

# Match each predicted field with its database reference field.
fields = {
    "Bedrooms": ("database_beds", "predicted_beds"),
    "Bathrooms": ("database_baths", "predicted_baths"),
    "Price": ("database_price", "predicted_price")
}

# Create an empty list for our results.
structured_results = []

# Evaluate each numeric field separately.
for name, (actual_col, predicted_col) in fields.items():

    # Convert the database values into numbers.
    # Invalid or missing values become NaN.
    actual = pd.to_numeric(
        structured_df[actual_col], errors="coerce"
    )

    # Convert the extracted predictions into numbers.
    predicted = pd.to_numeric(
        structured_df[predicted_col], errors="coerce"
    )

    # Find listings that contain a valid database value.
    valid = actual.notna()

    # Find listings that have both a reference and prediction.
    extracted = valid & predicted.notna()

    # Count correct predictions.
    correct = (actual[extracted] == predicted[extracted]).sum()

    # Calculate accuracy among listings with extracted values.
    accuracy = (
        correct / extracted.sum()
        if extracted.sum() > 0 else float("nan")
    )

    # Calculate extraction coverage.
    # This tells us how often the extractor returned a value.
    coverage = (
        extracted.sum() / valid.sum()
        if valid.sum() > 0 else float("nan")
    )

    # Store our results.
    structured_results.append({
        "Field": name,
        "Correct": int(correct),
        "Extracted": int(extracted.sum()),
        "Reference Available": int(valid.sum()),
        "Exact Match Accuracy": accuracy,
        "Coverage": coverage
    })

# Convert the results into a table.
structured_metrics_df = pd.DataFrame(structured_results)

# Display the results as percentages.
display(
    structured_metrics_df.style.format({
        "Exact Match Accuracy": "{:.2%}",
        "Coverage": "{:.2%}"
    })
)


,Field,Correct,Extracted,Reference Available,Exact Match Accuracy,Coverage
0,Bedrooms,16,18,25,88.89%,72.00%
1,Bathrooms,11,19,25,57.89%,76.00%
2,Price,0,1,25,0.00%,4.00%


In [16]:

# ============================================================
# STEP 13: OVERALL STRUCTURED-FIELD ACCURACY
# ============================================================

# Add up all correctly extracted values across the fields.
total_correct = structured_metrics_df["Correct"].sum()

# Add up all values that were extracted and evaluated.
total_extracted = structured_metrics_df["Extracted"].sum()

# Calculate overall accuracy for extracted structured values.
overall_structured_accuracy = (
    total_correct / total_extracted
    if total_extracted > 0 else float("nan")
)

# Display the overall accuracy.
print(f"Overall structured accuracy: {overall_structured_accuracy:.2%}")

# Compare the result against the 90% target.
if overall_structured_accuracy >= 0.90:
    print("90% exact-match target met for extracted values.")
else:
    print("90% exact-match target not met for extracted values.")


Overall structured accuracy: 71.05%
90% exact-match target not met for extracted values.


In [17]:

# ============================================================
# STEP 14: VALIDATE JSON FOR SEARCH INDEXING
# ============================================================

# Define the fields that every processed listing must contain.
required_fields = [
    "listing_id",
    "entities",
    "amenities",
    "condition_keywords",
    "financing_terms",
    "location_features"
]

# Define the fields that should contain lists of signals.
list_fields = [
    "amenities",
    "condition_keywords",
    "financing_terms",
    "location_features"
]

# Count how many records have a valid structure.
valid_records = 0

# Check every processed listing.
for listing in processed_listings:

    # Check that all required fields exist.
    has_fields = all(
        field in listing for field in required_fields
    )

    # Check that all signal categories contain lists of strings.
    valid_lists = all(
        isinstance(listing.get(field), list)
        and all(isinstance(x, str) for x in listing[field])
        for field in list_fields
    )

    # Check the structure of the Week 3 entity dictionary.
    valid_entities = isinstance(listing.get("entities"), dict)

    # Check whether the listing has a usable ID.
    valid_id = listing.get("listing_id") is not None

    # Count the record if all checks pass.
    if has_fields and valid_lists and valid_entities and valid_id:
        valid_records += 1

# Calculate the percentage of records with a valid schema.
schema_validity = (
    valid_records / len(processed_listings) * 100
    if processed_listings else 0
)

# Display the results.
print(f"Total listings: {len(processed_listings):,}")
print(f"Valid records: {valid_records:,}")
print(f"Schema validity: {schema_validity:.2f}%")


Total listings: 55,212
Valid records: 55,212
Schema validity: 100.00%


In [18]:

# ============================================================
# STEP 15: TEST SEARCH FILTERING
# ============================================================

# Create a function to filter properties using extracted signals.
def filter_listings(
    listings,
    amenity=None,
    condition=None,
    financing=None,
    location=None
):
    # Create an empty list for matching properties.
    results = []

    # Examine every processed property.
    for listing in listings:

        # Skip properties that do not contain the requested amenity.
        if amenity and amenity not in listing["amenities"]:
            continue

        # Skip properties that do not match the requested condition.
        if condition and condition not in listing["condition_keywords"]:
            continue

        # Skip properties that do not match the financing requirement.
        if financing and financing not in listing["financing_terms"]:
            continue

        # Skip properties that do not match the location requirement.
        if location and location not in listing["location_features"]:
            continue

        # The property satisfies all requested filters.
        results.append(listing)

    # Return all matching properties.
    return results


# Example: Find properties with pools that are renovated.
matching_listings = filter_listings(
    processed_listings,
    amenity="pool",
    condition="renovated"
)

# Display the number of matching listings.
print(f"Matching properties: {len(matching_listings):,}")

# Display the first five matching property IDs.
for listing in matching_listings[:5]:
    print(listing["listing_id"])


Matching properties: 4,836
1117039318
1119957879
1114484354
1111547604
1108761588


In [19]:
# ============================================================
# RELOAD THE UPDATED SIGNAL EXTRACTOR
# ============================================================

# Import importlib so Python can reload an edited script.
import importlib

# Import our Week 6 script from the scripts folder.
import scripts.signal_extractor as signal_module

# Reload the script to use our latest changes.
importlib.reload(signal_module)

# Recreate the Week 3 entity extractor.
entity_extractor = signal_module.EntityExtractor()

# Recreate the Week 6 signal extractor.
signal_extractor = signal_module.SignalExtractor(
    taxonomy=signal_module.SIGNAL_TAXONOMY,
    entity_extractor=entity_extractor
)

# Confirm the updated classes loaded successfully.
print("Updated EntityExtractor and SignalExtractor loaded!")

Updated EntityExtractor and SignalExtractor loaded!


In [20]:
# ============================================================
# TEST UPDATED EXTRACTION
# ============================================================

# Create sample descriptions using formats our old model missed.
test_descriptions = [
    "Beautiful 4-bed home with 2 1/2 baths.",
    "This home offers 2 full bathrooms and 1/2 bath.",
    "A charming 576-square-foot cabin.",
    "The property has 2,112 sf of living space.",
    "Central AC and an attached garage.",
    "The seller may finance this property.",
    "Located minutes from parks, schools, and shopping.",
    "Close to boutiques and restaurants.",
    "No swimming pool or garage."
]

# Test each description using the updated extractor.
for i, description in enumerate(test_descriptions, start=1):

    # Create a sample listing record.
    test_listing = {
        "L_ListingID": f"TEST{i}",
        "L_Remarks": description
    }

    # Extract all available signals.
    result = signal_extractor.extract_signals(test_listing)

    # Display the results.
    print(f"\nTEST {i}: {description}")
    print("Entities:", result["entities"])
    print("Amenities:", result["amenities"])
    print("Condition:", result["condition_keywords"])
    print("Financing:", result["financing_terms"])
    print("Location:", result["location_features"])


TEST 1: Beautiful 4-bed home with 2 1/2 baths.
Entities: {'bedrooms': 4, 'bathrooms': 2.5, 'price': None, 'sqft': None, 'amenities': []}
Amenities: []
Condition: []
Financing: []
Location: []

TEST 2: This home offers 2 full bathrooms and 1/2 bath.
Entities: {'bedrooms': None, 'bathrooms': 2.5, 'price': None, 'sqft': None, 'amenities': []}
Amenities: []
Condition: []
Financing: []
Location: []

TEST 3: A charming 576-square-foot cabin.
Entities: {'bedrooms': None, 'bathrooms': None, 'price': None, 'sqft': 576, 'amenities': []}
Amenities: []
Condition: []
Financing: []
Location: []

TEST 4: The property has 2,112 sf of living space.
Entities: {'bedrooms': None, 'bathrooms': None, 'price': None, 'sqft': 2112, 'amenities': []}
Amenities: []
Condition: []
Financing: []
Location: []

TEST 5: Central AC and an attached garage.
Entities: {'bedrooms': None, 'bathrooms': None, 'price': None, 'sqft': None, 'amenities': ['air_conditioning', 'garage']}
Amenities: ['air_conditioning', 'garage']
Co

In [21]:
# Import tools for reloading updated Python code.
import importlib
import scripts.signal_extractor as signal_module

# Reload our updated Python script.
importlib.reload(signal_module)

# Recreate the Week 3 extractor.
entity_extractor = signal_module.EntityExtractor()

# Recreate the Week 6 extractor.
signal_extractor = signal_module.SignalExtractor(
    taxonomy=signal_module.SIGNAL_TAXONOMY,
    entity_extractor=entity_extractor
)

# Confirm everything loaded.
print("Updated extractors initialized successfully!")

Updated extractors initialized successfully!


In [22]:
# ============================================================
# TEST UPDATED NEGATION AND LOCATION EXTRACTION
# ============================================================

# Define the descriptions and expected signal results.
test_cases = [
    {
        "text": "No swimming pool or garage.",
        "amenities": [],
        "location_features": []
    },
    {
        "text": "Without a patio or balcony.",
        "amenities": [],
        "location_features": []
    },
    {
        "text": "No pool, but an attached garage is available.",
        "amenities": ["garage"],
        "location_features": []
    },
    {
        "text": "A pool is not included.",
        "amenities": [],
        "location_features": []
    },
    {
        "text": "Near shopping, schools, parks, and restaurants.",
        "amenities": [],
        "location_features": [
            "near_schools",
            "near_parks",
            "near_shopping"
        ]
    },
    {
        "text": "Close to boutiques, parks, and local schools.",
        "amenities": [],
        "location_features": [
            "near_schools",
            "near_parks",
            "near_shopping"
        ]
    },
    {
        "text": "Beautiful home with a swimming pool and garage.",
        "amenities": ["pool", "garage"],
        "location_features": []
    }
]

# Count how many test cases pass.
passed = 0

# Run the extractor on every test description.
for i, test in enumerate(test_cases, start=1):

    # Create a temporary listing record.
    listing = {
        "L_ListingID": f"TEST{i}",
        "L_Remarks": test["text"]
    }

    # Extract the signals.
    result = signal_extractor.extract_signals(listing)

    # Compare amenities without depending on list order.
    amenities_correct = (
        set(result["amenities"]) == set(test["amenities"])
    )

    # Compare location features without depending on list order.
    location_correct = (
        set(result["location_features"])
        == set(test["location_features"])
    )

    # A test passes only if both categories are correct.
    correct = amenities_correct and location_correct

    if correct:
        passed += 1

    # Display the test results.
    print(f"Test {i}: {'PASS' if correct else 'FAIL'}")

    # Show details when a test fails.
    if not correct:
        print("  Description:", test["text"])
        print("  Actual amenities:", result["amenities"])
        print("  Expected amenities:", test["amenities"])
        print("  Actual location:", result["location_features"])
        print("  Expected location:", test["location_features"])

# Display the total number of successful tests.
print(f"\nTests passed: {passed}/{len(test_cases)}")

Test 1: FAIL
  Description: No swimming pool or garage.
  Actual amenities: ['pool', 'garage']
  Expected amenities: []
  Actual location: []
  Expected location: []
Test 2: FAIL
  Description: Without a patio or balcony.
  Actual amenities: ['patio', 'balcony']
  Expected amenities: []
  Actual location: []
  Expected location: []
Test 3: FAIL
  Description: No pool, but an attached garage is available.
  Actual amenities: ['pool', 'garage']
  Expected amenities: ['garage']
  Actual location: []
  Expected location: []
Test 4: PASS
Test 5: PASS
Test 6: PASS
Test 7: PASS

Tests passed: 4/7


In [24]:

# ============================================================
# STEP 16: LOAD PREVIOUSLY REVIEWED LISTINGS
# ============================================================

# Import the libraries needed for reading our evaluation data.
import pandas as pd
import json

# Find the CSV file containing our 25 completed manual reviews.
reviewed_file = output_dir / "week6_reviewed_evaluation.csv"

# Load the saved reviews into a DataFrame.
# This preserves the original predictions and manual labels.
reviewed_df = pd.read_csv(reviewed_file).fillna("")

# Confirm that the correct number of listings was loaded.
print(f"Manually reviewed listings: {len(reviewed_df)}")

# Display the first five listings and their manual labels.
display(
    reviewed_df[
        [
            "listing_id",
            "ground_truth_amenities",
            "ground_truth_condition",
            "ground_truth_financing",
            "ground_truth_location"
        ]
    ].head()
)


Manually reviewed listings: 25


,listing_id,ground_truth_amenities,ground_truth_condition,ground_truth_financing,ground_truth_location
0,1183036503,"[""patio"", ""garage"", ""laundry"", ""balcony""]",[],[],[]
1,1190152982,"[""updated_kitchen"", ""patio"", ""laundry"", ""parki...","[""updated"", ""move_in_ready""]",[],[]
2,1177518498,"[""parking""]",[],[],[]
3,1184980333,"[""balcony"", ""laundry"", ""parking""]",[],[],[]
4,1177243676,"[""patio"", ""pool"", ""fireplace"", ""laundry"", ""gar...",[],[],[]


In [25]:

# ============================================================
# STEP 17: GENERATE UPDATED EXTRACTION PREDICTIONS
# ============================================================

# Import importlib so our notebook can reload the latest script.
import importlib
import scripts.signal_extractor as signal_module

# Reload the current version of signal_extractor.py.
importlib.reload(signal_module)

# Recreate the updated Week 3 entity extractor.
entity_extractor = signal_module.EntityExtractor()

# Recreate the updated Week 6 signal extractor.
signal_extractor = signal_module.SignalExtractor(
    taxonomy=signal_module.SIGNAL_TAXONOMY,
    entity_extractor=entity_extractor
)

# Create a copy of our manually reviewed listings.
# This keeps the original evaluation data unchanged.
updated_df = reviewed_df.copy()

# Create an empty list to store updated predictions.
new_predictions = []

# Run our updated extractor on each reviewed listing.
for _, row in updated_df.iterrows():

    # Recreate the listing format expected by SignalExtractor.
    listing = {
        "L_ListingID": row["listing_id"],
        "L_Remarks": row["remarks"]
    }

    # Extract signals using the latest version of our script.
    result = signal_extractor.extract_signals(listing)

    # Save the complete result.
    new_predictions.append(result)

# Store the new results in separate columns.
#
# We use json.dumps() because the original predictions
# were also saved as JSON strings.
updated_df["new_entities"] = [
    json.dumps(result["entities"])
    for result in new_predictions
]

updated_df["new_amenities"] = [
    json.dumps(result["amenities"])
    for result in new_predictions
]

updated_df["new_condition"] = [
    json.dumps(result["condition_keywords"])
    for result in new_predictions
]

updated_df["new_financing"] = [
    json.dumps(result["financing_terms"])
    for result in new_predictions
]

updated_df["new_location"] = [
    json.dumps(result["location_features"])
    for result in new_predictions
]

# Save updated predictions to a separate CSV file.
# The original 25 manual reviews remain untouched.
updated_file = output_dir / "week6_updated_evaluation.csv"

updated_df.to_csv(updated_file, index=False)

# Confirm that the updated results were saved.
print(f"Updated predictions saved to: {updated_file.resolve()}")

# Preview the original and new amenity predictions.
display(
    updated_df[
        [
            "listing_id",
            "predicted_amenities",
            "new_amenities",
            "ground_truth_amenities"
        ]
    ].head()
)


Updated predictions saved to: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week6_updated_evaluation.csv


,listing_id,predicted_amenities,new_amenities,ground_truth_amenities
0,1183036503,"[""patio"", ""garage"", ""laundry"", ""balcony""]","[""patio"", ""garage"", ""laundry"", ""balcony""]","[""patio"", ""garage"", ""laundry"", ""balcony""]"
1,1190152982,"[""updated_kitchen"", ""patio"", ""laundry"", ""parki...","[""updated_kitchen"", ""patio"", ""laundry"", ""parki...","[""updated_kitchen"", ""patio"", ""laundry"", ""parki..."
2,1177518498,"[""parking""]","[""parking""]","[""parking""]"
3,1184980333,"[""balcony"", ""laundry"", ""parking""]","[""balcony"", ""laundry"", ""parking""]","[""balcony"", ""laundry"", ""parking""]"
4,1177243676,"[""patio"", ""pool"", ""fireplace"", ""laundry"", ""gar...","[""patio"", ""pool"", ""fireplace"", ""laundry"", ""gar...","[""patio"", ""pool"", ""fireplace"", ""laundry"", ""gar..."


In [26]:

# ============================================================
# STEP 18: COMPARE ORIGINAL AND UPDATED EXTRACTION METRICS
# ============================================================

# Import classification metrics.
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# Match the original and updated prediction columns
# with the manually verified ground-truth columns.
categories = {
    "Amenities": (
        "predicted_amenities",
        "new_amenities",
        "ground_truth_amenities"
    ),
    "Condition": (
        "predicted_condition",
        "new_condition",
        "ground_truth_condition"
    ),
    "Financing": (
        "predicted_financing",
        "new_financing",
        "ground_truth_financing"
    ),
    "Location": (
        "predicted_location",
        "new_location",
        "ground_truth_location"
    )
}

# Create an empty list for comparison results.
comparison_results = []

# Track all individual signal decisions for overall scores.
overall_counts = {
    "Original": {"TP": 0, "FP": 0, "FN": 0},
    "Updated": {"TP": 0, "FP": 0, "FN": 0}
}

# Evaluate each category individually.
for category, (old_col, new_col, truth_col) in categories.items():

    # Evaluate both versions of the extractor.
    for version, prediction_col in [
        ("Original", old_col),
        ("Updated", new_col)
    ]:

        # Store correct, incorrect, and missed signal counts.
        tp = 0
        fp = 0
        fn = 0

        # Examine each manually reviewed listing.
        for _, row in updated_df.iterrows():

            # Convert saved JSON text into Python sets.
            # Sets make comparing labels straightforward.
            predicted = set(json.loads(row[prediction_col]))
            actual = set(json.loads(row[truth_col]))

            # Correctly predicted signals.
            tp += len(predicted & actual)

            # Incorrectly predicted signals.
            fp += len(predicted - actual)

            # Actual signals the extractor missed.
            fn += len(actual - predicted)

        # Calculate micro-precision for this category.
        precision = tp / (tp + fp) if tp + fp > 0 else float("nan")

        # Calculate micro-recall for this category.
        recall = tp / (tp + fn) if tp + fn > 0 else float("nan")

        # Calculate micro-F1 using the signal counts.
        f1 = (
            2 * tp / (2 * tp + fp + fn)
            if 2 * tp + fp + fn > 0 else float("nan")
        )

        # Add the category counts to the overall totals.
        overall_counts[version]["TP"] += tp
        overall_counts[version]["FP"] += fp
        overall_counts[version]["FN"] += fn

        # Store the category results.
        comparison_results.append({
            "Category": category,
            "Version": version,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn
        })

# Calculate overall micro-averaged metrics.
for version, counts in overall_counts.items():

    tp = counts["TP"]
    fp = counts["FP"]
    fn = counts["FN"]

    precision = tp / (tp + fp) if tp + fp > 0 else float("nan")
    recall = tp / (tp + fn) if tp + fn > 0 else float("nan")

    f1 = (
        2 * tp / (2 * tp + fp + fn)
        if 2 * tp + fp + fn > 0 else float("nan")
    )

    # Add the overall row to the comparison.
    comparison_results.append({
        "Category": "Overall (Micro)",
        "Version": version,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "TP": tp,
        "FP": fp,
        "FN": fn
    })

# Convert the results into a DataFrame.
comparison_df = pd.DataFrame(comparison_results)

# Display all category and overall scores.
display(
    comparison_df.style.format({
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1 Score": "{:.2%}"
    })
)


,Category,Version,Precision,Recall,F1 Score,TP,FP,FN
0,Amenities,Original,100.00%,90.79%,95.17%,69,0,7
1,Amenities,Updated,100.00%,96.05%,97.99%,73,0,3
2,Condition,Original,100.00%,78.57%,88.00%,11,0,3
3,Condition,Updated,91.67%,78.57%,84.62%,11,1,3
4,Financing,Original,nan%,0.00%,0.00%,0,0,2
5,Financing,Updated,nan%,0.00%,0.00%,0,0,2
6,Location,Original,100.00%,20.00%,33.33%,3,0,12
7,Location,Updated,88.24%,100.00%,93.75%,15,2,0
8,Overall (Micro),Original,100.00%,77.57%,87.37%,83,0,24
9,Overall (Micro),Updated,97.06%,92.52%,94.74%,99,3,8


In [27]:

# ============================================================
# STEP 20: IDENTIFY REMAINING EXTRACTION ERRORS
# ============================================================

import json
import pandas as pd

# Match each signal category to its updated predictions
# and manually verified ground-truth labels.
categories = {
    "Amenities": ("new_amenities", "ground_truth_amenities"),
    "Condition": ("new_condition", "ground_truth_condition"),
    "Financing": ("new_financing", "ground_truth_financing"),
    "Location": ("new_location", "ground_truth_location")
}

# Store listing-level extraction errors.
errors = []

# Examine each manually reviewed listing.
for _, row in updated_df.iterrows():

    # Compare predictions with ground truth for every category.
    for category, (prediction_col, truth_col) in categories.items():

        # Convert JSON strings into Python sets.
        predicted = set(json.loads(row[prediction_col]))
        actual = set(json.loads(row[truth_col]))

        # False positives are predicted signals that should not exist.
        false_positives = predicted - actual

        # False negatives are correct signals the extractor missed.
        false_negatives = actual - predicted

        # Only save rows where an extraction error occurred.
        if false_positives or false_negatives:

            errors.append({
                "listing_id": row["listing_id"],
                "category": category,
                "false_positives": ", ".join(sorted(false_positives)),
                "false_negatives": ", ".join(sorted(false_negatives)),
                "remarks": row["remarks"]
            })

# Convert the errors into a DataFrame.
errors_df = pd.DataFrame(errors)

# Display the remaining errors with the full listing remarks.
pd.set_option("display.max_colwidth", 300)

display(errors_df)

# Save errors so they can be reviewed outside the notebook.
error_file = output_dir / "week6_remaining_errors.csv"

errors_df.to_csv(error_file, index=False)

print(f"Remaining error records: {len(errors_df)}")
print(f"Saved to: {error_file}")


,listing_id,category,false_positives,false_negatives,remarks
0,1176619756,Location,near_parks,,"Desirable upstairs 2BR/2BA condo in the Verano community of La Jolla Colony. Bright and well-maintained with granite countertops, stainless steel appliances, and a combination of tile and laminate flooring throughout. The home also features in-unit laundry and ceiling fans in both bedrooms for a..."
1,1179206401,Amenities,,"air_conditioning, updated_kitchen","Modern luxury meets iconic mid-century architecture in this beautifully renovated 3-bedroom, 3-bath Sandpiper residence, tucked within the exclusive Circle 17 enclave. Offering 2,372 square feet, the home pairs contemporary finishes with the effortless indoor-outdoor lifestyle that defines Palm ..."
2,1179206401,Condition,,updated,"Modern luxury meets iconic mid-century architecture in this beautifully renovated 3-bedroom, 3-bath Sandpiper residence, tucked within the exclusive Circle 17 enclave. Offering 2,372 square feet, the home pairs contemporary finishes with the effortless indoor-outdoor lifestyle that defines Palm ..."
3,1182292146,Condition,fixer_upper,,"Prime commercial opportunity on Nielson Road in Phelan! Located on Nielson Rd just off Sheep Creek Rd, this high-potential commercial parcel offers unbeatable proximity to schools, shopping, and main transit routes. Includes an existing 2BD/2BA manufactured home on-site (needs TLC, no value attr..."
4,1182292146,Financing,,fixer_upper,"Prime commercial opportunity on Nielson Road in Phelan! Located on Nielson Rd just off Sheep Creek Rd, this high-potential commercial parcel offers unbeatable proximity to schools, shopping, and main transit routes. Includes an existing 2BD/2BA manufactured home on-site (needs TLC, no value attr..."
5,1190553811,Location,near_parks,,"Charming California bungalow tucked into one of Los Feliz's most sought-after residential pockets, just moments from the Village, Griffith Park trails, and neighborhood cafes and shops. This 2-bed, 2-bath home blends timeless character with serene indoor-outdoor living. The main living spaces in..."
6,1179681062,Condition,,fixer_upper,"Handyman fixer on 3.4 acres. The home has great potential with 3 bedrooms and 2 baths, enclosed patio, covered porch deck, covered back patio deck, storage sheds, one car garage, fenced side yard for pets, and plenty of space for your trucks and more."
7,1191022507,Condition,,fixer_upper,"Step into soaring vaulted ceilings in the heart of Downtown San Diego—this is loft living at its most dramatic. Sun, space, and architectural scale you simply can't find in a standard unit. The generous footprint already lives like a two-bedroom, true to authentic loft style with no legally defi..."
8,1179687969,Amenities,,air_conditioning,"Welcome to Villa Mira, where comfort, style, and convenience come together in one of the community's most desirable upper-level residences. Perfectly situated with abundant guest parking nearby, this beautifully updated three-bedroom, two-bath home offers an exceptional blend of privacy, natural..."
9,1179687969,Financing,,move_in_ready,"Welcome to Villa Mira, where comfort, style, and convenience come together in one of the community's most desirable upper-level residences. Perfectly situated with abundant guest parking nearby, this beautifully updated three-bedroom, two-bath home offers an exceptional blend of privacy, natural..."


Remaining error records: 10
Saved to: c:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week6_remaining_errors.csv


In [31]:

# ============================================================
# STEP 21: COMPARE STRUCTURED EXTRACTION ACCURACY
# ============================================================

import pandas as pd
import json

# Identify the numeric fields we want to evaluate.
# Each field is matched against its reference database column.
structured_fields = {
    "Bedrooms": ("bedrooms", "database_beds"),
    "Bathrooms": ("bathrooms", "database_baths"),
    "Price": ("price", "database_price")
}

# Define the original and updated prediction columns.
versions = {
    "Original": "predicted_entities",
    "Updated": "new_entities"
}

# Create an empty list to store our evaluation results.
results = []

# Evaluate the original and updated extractors separately.
for version, prediction_column in versions.items():

    # Convert the JSON strings into Python dictionaries.
    extracted_entities = updated_df[prediction_column].apply(
        json.loads
    )

    # Evaluate each structured field.
    for field_name, (entity_key, database_column) in structured_fields.items():

        # Get the values predicted by the extractor.
        predicted = pd.to_numeric(
            extracted_entities.apply(
                lambda entities: entities.get(entity_key)
            ),
            errors="coerce"
        )

        # Get the actual values stored in the database.
        actual = pd.to_numeric(
            updated_df[database_column],
            errors="coerce"
        )

        # Identify listings with a database reference value.
        reference_available = actual.notna()

        # Identify listings with both a reference and prediction.
        extracted = reference_available & predicted.notna()

        # Count the number of exact matches.
        correct = int(
            (predicted[extracted] == actual[extracted]).sum()
        )

        # Count the number of available predictions.
        extracted_count = int(extracted.sum())

        # Count the number of available reference values.
        reference_count = int(reference_available.sum())

        # Calculate exact-match accuracy among extracted values.
        accuracy = (
            correct / extracted_count
            if extracted_count > 0 else float("nan")
        )

        # Calculate extraction coverage against database references.
        coverage = (
            extracted_count / reference_count
            if reference_count > 0 else float("nan")
        )

        # Save the results for this field and version.
        results.append({
            "Version": version,
            "Field": field_name,
            "Correct": correct,
            "Extracted": extracted_count,
            "Reference Available": reference_count,
            "Exact Match Accuracy": accuracy,
            "Coverage": coverage
        })

# Convert our evaluation results into a DataFrame.
structured_results_df = pd.DataFrame(results)

# Display accuracy and coverage for every numeric field.
print("STRUCTURED FIELD COMPARISON")

display(
    structured_results_df.style.format({
        "Exact Match Accuracy": "{:.2%}",
        "Coverage": "{:.2%}"
    })
)

# ============================================================
# CALCULATE OVERALL STRUCTURED ACCURACY
# ============================================================

# Sum correct and extracted values for each model version.
overall_structured_df = (
    structured_results_df
    .groupby("Version")[["Correct", "Extracted", "Reference Available"]]
    .sum()
)

# Calculate combined exact-match accuracy.
overall_structured_df["Accuracy"] = (
    overall_structured_df["Correct"]
    / overall_structured_df["Extracted"]
)

# Calculate combined extraction coverage.
overall_structured_df["Coverage"] = (
    overall_structured_df["Extracted"]
    / overall_structured_df["Reference Available"]
)

# Show the overall comparison.
print("\nOVERALL STRUCTURED ACCURACY")

display(
    overall_structured_df.style.format({
        "Accuracy": "{:.2%}",
        "Coverage": "{:.2%}"
    })
)

# ============================================================
# CHECK AGAINST THE 90% BENCHMARK
# ============================================================

# Retrieve the updated accuracy.
updated_accuracy = overall_structured_df.loc[
    "Updated", "Accuracy"
]

# Display whether the updated extractor meets the target.
if updated_accuracy >= 0.90:
    print("\n90% exact-match benchmark met!")
else:
    print("\n90% exact-match benchmark not yet met.")

# Save the results separately from the manual evaluation file.
structured_output_file = (
    output_dir / "week6_structured_comparison.csv"
)

structured_results_df.to_csv(
    structured_output_file,
    index=False
)

print(f"Results saved to: {structured_output_file}")


STRUCTURED FIELD COMPARISON


,Version,Field,Correct,Extracted,Reference Available,Exact Match Accuracy,Coverage
0,Original,Bedrooms,16,18,25,88.89%,72.00%
1,Original,Bathrooms,11,19,25,57.89%,76.00%
2,Original,Price,0,1,25,0.00%,4.00%
3,Updated,Bedrooms,16,18,25,88.89%,72.00%
4,Updated,Bathrooms,11,19,25,57.89%,76.00%
5,Updated,Price,0,1,25,0.00%,4.00%



OVERALL STRUCTURED ACCURACY


,Correct,Extracted,Reference Available,Accuracy,Coverage
Version,,,,,
Original,27,38,75,71.05%,50.67%
Updated,27,38,75,71.05%,50.67%



90% exact-match benchmark not yet met.
Results saved to: c:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week6_structured_comparison.csv


In [32]:

# ============================================================
# IDENTIFY STRUCTURED EXTRACTION ERRORS
# ============================================================

import json
import pandas as pd

# Define the fields and database reference columns.
fields = {
    "Bedrooms": ("bedrooms", "database_beds"),
    "Bathrooms": ("bathrooms", "database_baths"),
    "Price": ("price", "database_price")
}

# Store the listings where extracted values disagree
# with the database or are missing.
structured_errors = []

# Examine each previously reviewed listing.
for _, row in updated_df.iterrows():

    # Convert the updated entities from JSON into a dictionary.
    entities = json.loads(row["new_entities"])

    # Compare each field against the database.
    for field_name, (entity_key, reference_col) in fields.items():

        predicted = entities.get(entity_key)
        actual = row[reference_col]

        # Skip rows without a valid database reference.
        if pd.isna(actual) or actual == "":
            continue

        # Distinguish missing values from incorrect values.
        if predicted is None:
            error_type = "Missing"

        elif float(predicted) != float(actual):
            error_type = "Mismatch"

        else:
            # Correct predictions do not need investigation.
            continue

        # Save the listing and its error.
        structured_errors.append({
            "listing_id": row["listing_id"],
            "field": field_name,
            "predicted": predicted,
            "database_value": actual,
            "error_type": error_type,
            "remarks": row["remarks"]
        })

# Create a DataFrame containing the identified issues.
structured_errors_df = pd.DataFrame(structured_errors)

# Display the errors for manual inspection.
pd.set_option("display.max_colwidth", 250)

display(structured_errors_df)

# Save a CSV containing the errors.
error_file = output_dir / "week6_structured_errors.csv"

structured_errors_df.to_csv(error_file, index=False)

print(f"Structured errors identified: {len(structured_errors_df)}")
print(f"Saved to: {error_file}")


,listing_id,field,predicted,database_value,error_type,remarks
0,1183036503,Bathrooms,2.5,3.0,Mismatch,"Welcome to this beautifully appointed Spanish-style condo offering 3 bedrooms, 2.5 bathrooms, and 1,591 square feet of thoughtfully designed living space which was the model home from the builder. A spacious front patio welcomes you to the home, ..."
1,1183036503,Price,NaN,749900.0,Missing,"Welcome to this beautifully appointed Spanish-style condo offering 3 bedrooms, 2.5 bathrooms, and 1,591 square feet of thoughtfully designed living space which was the model home from the builder. A spacious front patio welcomes you to the home, ..."
2,1190152982,Bedrooms,2.0,3.0,Mismatch,"Back on the market with New Pergo Flooring and Paint!!! A rare opportunity in Upper Yucaipa—two homes on one lot with a versatile layout and thoughtful updates throughout! 35263 Beech Avenue offers a charming 2-bedroom, 1-bath front home plus a s..."
3,1190152982,Bathrooms,1.0,2.0,Mismatch,"Back on the market with New Pergo Flooring and Paint!!! A rare opportunity in Upper Yucaipa—two homes on one lot with a versatile layout and thoughtful updates throughout! 35263 Beech Avenue offers a charming 2-bedroom, 1-bath front home plus a s..."
4,1190152982,Price,NaN,574900.0,Missing,"Back on the market with New Pergo Flooring and Paint!!! A rare opportunity in Upper Yucaipa—two homes on one lot with a versatile layout and thoughtful updates throughout! 35263 Beech Avenue offers a charming 2-bedroom, 1-bath front home plus a s..."
5,1177518498,Bathrooms,1.0,2.0,Mismatch,"**TWO FLAT PARCELS ARE INCLUDED IN THE SALES PRICE!** 369 Log Lane, featuring the charming mountain cabin, and the adjoining vacant parcel at 375 Log Lane are being sold together, offering approximately 8,300 square feet of combined land. Two fla..."
6,1177518498,Price,NaN,234900.0,Missing,"**TWO FLAT PARCELS ARE INCLUDED IN THE SALES PRICE!** 369 Log Lane, featuring the charming mountain cabin, and the adjoining vacant parcel at 375 Log Lane are being sold together, offering approximately 8,300 square feet of combined land. Two fla..."
7,1184980333,Bathrooms,2.5,3.0,Mismatch,"Priced to sell !! Bright and Spacious 3BR + 2.5BA condo in the sought-after Alcott Terrace complex, located in the heart of Pico-Robertson. Sophisticated, Elegant open Lay-out and spacious floor plan with high ceilings. Unit Features crown mol..."
8,1184980333,Price,NaN,895000.0,Missing,"Priced to sell !! Bright and Spacious 3BR + 2.5BA condo in the sought-after Alcott Terrace complex, located in the heart of Pico-Robertson. Sophisticated, Elegant open Lay-out and spacious floor plan with high ceilings. Unit Features crown mol..."
9,1177243676,Bathrooms,2.5,3.0,Mismatch,"Located in San Martin, this gorgeous and peaceful light and bright open floor plan, 4-bed, 2 1/2-bath home offers 2,112 sf of living space on a spacious lot of approximately 2.7 acres. Relaxing patio great for morning coffee over looking the beau..."


Structured errors identified: 48
Saved to: c:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week6_structured_errors.csv


In [36]:

# ============================================================
# STEP 22: INTERACTIVE STRUCTURED EXTRACTION REVIEW
# ============================================================

# This interface works like our previous free-text review tool.
#
# For each listing:
# 1. Read the full listing description.
# 2. See the model's extracted numeric values.
# 3. Correct the values directly in the input boxes.
# 4. Leave a box empty if the value is not clearly stated.
# 5. Click Save & Next to continue.
#
# Previously saved reviews will be loaded automatically.

import pandas as pd
import json
import html
import ipywidgets as widgets

from IPython.display import display, clear_output, HTML


# ============================================================
# 1. LOAD EXISTING REVIEWS OR CREATE A NEW REVIEW FILE
# ============================================================

structured_review_file = (
    output_dir / "week6_structured_manual_review.csv"
)

# The fields we want to evaluate.
structured_fields = [
    "bedrooms",
    "bathrooms",
    "price",
    "sqft"
]

if structured_review_file.exists():

    # Continue from previously saved manual reviews.
    structured_manual_df = pd.read_csv(
        structured_review_file,
        keep_default_na=False
    )

    print("Previous manual reviews loaded!")

else:

    # Use the same 25 listings we previously reviewed.
    structured_manual_df = (
        updated_df.copy().reset_index(drop=True)
    )

    # Create empty columns for manual corrections.
    for field in structured_fields:
        structured_manual_df[f"truth_{field}"] = ""

    # Track which listings have been completed.
    structured_manual_df["structured_reviewed"] = False

    # Save whether a field is actually stated in the remarks.
    for field in structured_fields:
        structured_manual_df[f"known_{field}"] = False

    print("New structured review created!")


# ============================================================
# 2. CREATE INPUT WIDGETS
# ============================================================

# Store the editable numeric fields.
structured_inputs = {}

for field in structured_fields:

    structured_inputs[field] = widgets.Text(
        placeholder="Enter correct value or leave blank",
        layout=widgets.Layout(width="350px")
    )

# Display listing details here.
review_output = widgets.Output(
    layout=widgets.Layout(width="100%")
)

# Create navigation buttons.
previous_button = widgets.Button(
    description="Previous"
)

save_next_button = widgets.Button(
    description="Save & Next",
    button_style="success"
)

next_button = widgets.Button(
    description="Next"
)

# Allow jumping directly to a specific listing.
jump_input = widgets.BoundedIntText(
    value=1,
    min=1,
    max=len(structured_manual_df),
    description="Listing #"
)

jump_button = widgets.Button(
    description="Go"
)


# ============================================================
# 3. FIND WHERE WE LEFT OFF
# ============================================================

# Identify listings that have not been reviewed.
reviewed_status = (
    structured_manual_df["structured_reviewed"]
    .astype(str)
    .str.lower()
    .eq("true")
)

unreviewed = [
    i for i, status in enumerate(reviewed_status)
    if not status
]

# Continue with the first incomplete listing.
current_structured_index = (
    unreviewed[0] if unreviewed else 0
)


# ============================================================
# 4. DISPLAY A LISTING
# ============================================================

def display_structured_review():
    """
    Show the listing remarks and editable ground-truth fields.
    """

    row = structured_manual_df.iloc[current_structured_index]

    # Load the updated model predictions.
    predictions = json.loads(row["new_entities"])

    with review_output:

        clear_output(wait=True)

        print(
            f"Listing {current_structured_index + 1} "
            f"of {len(structured_manual_df)}"
        )

        print(f"Listing ID: {row['listing_id']}")

        # ----------------------------------------------------
        # DISPLAY FULL REMARKS WITH AUTOMATIC WRAPPING
        # ----------------------------------------------------

        remarks = html.escape(str(row["remarks"]))

        display(HTML(
            '<div style="'
            'white-space: pre-wrap; '
            'overflow-wrap: anywhere; '
            'font-family: Arial, sans-serif; '
            'font-size: 14px; '
            'line-height: 1.7; '
            'padding: 12px; '
            'max-width: 100%;'
            '">'
            + remarks +
            '</div>'
        ))

        # ----------------------------------------------------
        # DISPLAY EDITABLE FIELDS
        # ----------------------------------------------------

        for field in structured_fields:

            predicted_value = predictions.get(field)

            # Display the original model prediction.
            print(f"\n{field.upper()}")
            print(f"Predicted: {predicted_value}")

            # Determine whether this listing was reviewed before.
            was_reviewed = (
                str(row["structured_reviewed"]).lower()
                == "true"
            )

            # Previously reviewed values take priority.
            if was_reviewed:

                saved_value = row[f"truth_{field}"]

                structured_inputs[field].value = str(
                    saved_value
                ).strip()

            else:

                # Pre-fill with the model's predictions
                # so the user only changes incorrect fields.
                structured_inputs[field].value = (
                    str(predicted_value)
                    if predicted_value is not None else ""
                )

            # Display the editable input box.
            display(structured_inputs[field])

        # ----------------------------------------------------
        # DISPLAY REVIEW STATUS
        # ----------------------------------------------------

        reviewed = (
            str(row["structured_reviewed"]).lower()
            == "true"
        )

        print(
            "\nStatus:",
            "Reviewed" if reviewed else "Not yet reviewed"
        )

        print(
            "\nLeave a field blank if the remarks do not "
            "provide a clear property-wide value."
        )


# ============================================================
# 5. SAVE CURRENT CORRECTIONS
# ============================================================

def save_structured_review():
    """
    Save corrected structured values for the current listing.
    """

    row_index = structured_manual_df.index[
        current_structured_index
    ]

    # Store the corrected values for each field.
    for field in structured_fields:

        entered_value = structured_inputs[field].value.strip()

        structured_manual_df.at[
            row_index,
            f"truth_{field}"
        ] = entered_value

        # A nonempty manual value means the field is verifiable.
        structured_manual_df.at[
            row_index,
            f"known_{field}"
        ] = bool(entered_value)

    # Mark the listing as completed.
    structured_manual_df.at[
        row_index,
        "structured_reviewed"
    ] = True

    # Save progress immediately.
    structured_manual_df.to_csv(
        structured_review_file,
        index=False
    )


# ============================================================
# 6. NAVIGATION BUTTONS
# ============================================================

def go_previous(button):

    global current_structured_index

    # Save before moving to another listing.
    save_structured_review()

    if current_structured_index > 0:
        current_structured_index -= 1

    display_structured_review()


def save_and_next(button):

    global current_structured_index

    # Save the current listing's corrections.
    save_structured_review()

    if current_structured_index < len(structured_manual_df) - 1:
        current_structured_index += 1

    display_structured_review()


def go_next(button):

    global current_structured_index

    # Save before navigating forward.
    save_structured_review()

    if current_structured_index < len(structured_manual_df) - 1:
        current_structured_index += 1

    display_structured_review()


def jump_to_listing(button):

    global current_structured_index

    # Save the current listing before jumping.
    save_structured_review()

    # Convert the displayed listing number into a Python index.
    current_structured_index = jump_input.value - 1

    display_structured_review()


# ============================================================
# 7. CONNECT AND DISPLAY THE INTERFACE
# ============================================================

previous_button.on_click(go_previous)

save_next_button.on_click(save_and_next)

next_button.on_click(go_next)

jump_button.on_click(jump_to_listing)

# Display the navigation buttons.
display(
    widgets.HBox([
        previous_button,
        save_next_button,
        next_button
    ])
)

# Display the listing jump controls.
display(
    widgets.HBox([
        jump_input,
        jump_button
    ])
)

# Display the full review interface.
display(review_output)

# Show the first incomplete listing.
display_structured_review()


New structured review created!


Output(layout=Layout(width='100%'))

In [37]:

# ============================================================
# STEP 23: CALCULATE FINAL STRUCTURED EXTRACTION ACCURACY
# ============================================================

import pandas as pd
import json

# Load our completed manual structured reviews.
review_file = output_dir / "week6_structured_manual_review.csv"

evaluation = pd.read_csv(
    review_file,
    keep_default_na=False
)

# Keep only listings that have been manually reviewed.
evaluation = evaluation[
    evaluation["structured_reviewed"].astype(str).str.lower() == "true"
].copy()

# Define the four structured fields from Week 3.
fields = ["bedrooms", "bathrooms", "price", "sqft"]

# Store results for each field.
results = []

# Track overall totals.
total_correct = 0
total_verifiable = 0
total_extracted = 0
total_unsupported = 0

# Compare predictions with your manually verified values.
for field in fields:

    correct = 0
    verifiable = 0
    extracted = 0
    unsupported = 0

    for _, row in evaluation.iterrows():

        # Get the manually corrected value.
        truth_text = str(row[f"truth_{field}"]).strip()

        # Get the updated model's prediction.
        predicted_entities = json.loads(row["new_entities"])
        prediction = predicted_entities.get(field)

        # If the field is not stated in the remarks,
        # a numeric prediction is considered unsupported.
        if not truth_text:

            if prediction is not None:
                unsupported += 1

            continue

        # Convert the manually verified number to numeric form.
        truth = float(truth_text.replace(",", ""))

        # Every verifiable value contributes to the denominator.
        verifiable += 1

        # Count predictions and correct matches.
        if prediction is not None:

            extracted += 1

            if float(prediction) == truth:
                correct += 1

    # Calculate accuracy and coverage.
    accuracy = correct / verifiable if verifiable else float("nan")
    coverage = extracted / verifiable if verifiable else float("nan")

    # Save field-level results.
    results.append({
        "Field": field.title(),
        "Correct": correct,
        "Verifiable": verifiable,
        "Extracted": extracted,
        "Unsupported Predictions": unsupported,
        "Accuracy": accuracy,
        "Coverage": coverage
    })

    # Add to overall totals.
    total_correct += correct
    total_verifiable += verifiable
    total_extracted += extracted
    total_unsupported += unsupported

# Create a summary table.
results_df = pd.DataFrame(results)

print(f"Listings manually reviewed: {len(evaluation)}")

display(
    results_df.style.format({
        "Accuracy": "{:.2%}",
        "Coverage": "{:.2%}"
    })
)

# ============================================================
# OVERALL STRUCTURED EXTRACTION PERFORMANCE
# ============================================================

# Calculate recall-like exact-match accuracy for fields
# with verifiable ground-truth values.
overall_accuracy = (
    total_correct / total_verifiable
    if total_verifiable else float("nan")
)

# Include unsupported predictions as errors in an
# additional strict extraction score.
strict_denominator = total_verifiable + total_unsupported

strict_accuracy = (
    total_correct / strict_denominator
    if strict_denominator else float("nan")
)

# Display the final results.
print("\nFINAL STRUCTURED EXTRACTION RESULTS")
print(f"Correct extractions: {total_correct}")
print(f"Verifiable fields: {total_verifiable}")
print(f"Unsupported predictions: {total_unsupported}")

print(f"\nVerifiable-field accuracy: {overall_accuracy:.2%}")
print(f"Strict extraction score: {strict_accuracy:.2%}")

# Check the stricter score against the Week 6 target.
if strict_denominator == 0:
    print("\nNo fields available for evaluation.")
elif strict_accuracy >= 0.90:
    print("\n90% structured extraction benchmark MET!")
else:
    print("\n90% structured extraction benchmark NOT YET MET.")

# Save your final evaluation table.
results_df.to_csv(
    output_dir / "week6_structured_manual_scores.csv",
    index=False
)

print("\nStructured evaluation saved successfully!")


Listings manually reviewed: 25


,Field,Correct,Verifiable,Extracted,Unsupported Predictions,Accuracy,Coverage
0,Bedrooms,18,20,18,0,90.00%,90.00%
1,Bathrooms,19,19,19,0,100.00%,100.00%
2,Price,0,0,0,1,nan%,nan%
3,Sqft,10,10,10,0,100.00%,100.00%



FINAL STRUCTURED EXTRACTION RESULTS
Correct extractions: 47
Verifiable fields: 49
Unsupported predictions: 1

Verifiable-field accuracy: 95.92%
Strict extraction score: 94.00%

90% structured extraction benchmark MET!

Structured evaluation saved successfully!
